In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from dotenv import load_dotenv

In [ ]:

load_dotenv(override=True)
API_KEY = os.getenv('ALPHAVANTAGE_API_KEY')

In [13]:
#Alpha Vantage API example to get stock price from AAPL
URL = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol=AAPL&apikey={API_KEY}"
response = requests.get(URL)
data = response.json()
price=data["Global Quote"]["05. price"]
print(f"The current price of AAPL is: ${float(price):.2f}")

The current price of AAPL is: $271.49


In [33]:
# Get Apple's CIK first
ticker_url = "https://www.sec.gov/files/company_tickers.json"
headers = {'User-Agent': 'StudentProject student@email.com'} 
ticker_response = requests.get(ticker_url, headers=headers)
ticker_data = ticker_response.json()
cik = None
for company in ticker_data.values():
    if company['ticker']=='AAPL':
        cik=str(company['cik_str']).zfill(10)
        break
print(f"Apple's Cik: {cik}")


# Now get financial data
facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
facts_response = requests.get(facts_url, headers=headers)
facts_data = facts_response.json()


Apple's Cik: 0000320193


In [30]:
#This tag RevenueFromContractWithCustomerExcludingAssessedTax is chosen due to a new accounting standard  (ASC 606) adopted in 2018, 
#Revenues used to be the old but it might have inluded the tax in some reports, this is the new way to report
revenue_data=facts_data['facts']['us-gaap']['RevenueFromContractWithCustomerExcludingAssessedTax']['units']['USD']
annual_reports=[r for r in revenue_data if r['form']=='10-K']
latest_revenue=sorted(annual_reports, key=lambda x:x['end'], reverse=True)[0]
revenue=latest_revenue['val']
print(f"Apple's latest revenue:${revenue:,.0f}")



Apple's latest revenue:$416,161,000,000


In [31]:
overview_url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol=AAPL&apikey={API_KEY}"
overview_response = requests.get(overview_url)
overview_data = overview_response.json()


overview_data

{'Symbol': 'AAPL',
 'AssetType': 'Common Stock',
 'Name': 'Apple Inc',
 'Description': 'Apple Inc. is a leading American multinational technology company that specializes in innovative consumer electronics, software, and online services. With a record revenue of $274.5 billion in 2020, it holds the title of the world\'s most valuable publicly traded company and is a dominant force in the global technology landscape. Its flagship products, such as the iPhone, iPad, and Mac, have cemented its reputation as a trailblazer in the sector, positioning it as the fourth-largest PC vendor and smartphone manufacturer worldwide. As a cornerstone of the "Big Five" technology companies, Apple continues to set industry standards and drive advancements in technology and consumer engagement.',
 'CIK': '320193',
 'Exchange': 'NASDAQ',
 'Currency': 'USD',
 'Country': 'USA',
 'Sector': 'TECHNOLOGY',
 'Industry': 'CONSUMER ELECTRONICS',
 'Address': 'ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STATES, 95014',

In [32]:
market_cap = float(overview_data['MarketCapitalization'])

print(f"Apple's market cap: ${market_cap:,.0f}")

# Calculate P/S Ratio
ps_ratio = market_cap / revenue

print(f"\nPrice-to-Sales Ratio: {ps_ratio:.2f}")

Apple's market cap: $4,029,017,227,000

Price-to-Sales Ratio: 9.68
